<a href="https://colab.research.google.com/github/3iqpotato/softuni_course_project/blob/main/softuni_exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai pypdf chromadb elevenlabs langchain-text-splitters

#Imports

In [ ]:
from enum import Enum
from google.colab import userdata
import json

from openai import OpenAI
from pydantic import BaseModel, Field

import chromadb
from pypdf import PdfReader


from elevenlabs import ElevenLabs

from IPython.display import display, Audio, Image
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

#Requirements and constants

In [ ]:
SECRET_NAMES = {
    "OPENAI_API_KEY": "Open_AI_Free_Key",
    "ELEVENLABS_API_KEY": "Eleven_labs_api_key",
}


MODELS = {
    "Embedding_Model": "text-embedding-3-small",
    "Gpt_4o_model": "gpt-4o-mini"

}

#Helping Functions

In [ ]:
def load_api_keys(secret_names: dict) -> dict:

    keys = {}
    for internal_name, secret_name in secret_names.items():
        try:
            value = userdata.get(secret_name)
        except userdata.SecretNotFoundError:
            raise EnvironmentError(
                f"Secret '{secret_name}' не е намерен в Colab Secrets. "
                f"Провери дали името е точно такова (иконата с ключ вляво)."
            )
        except userdata.NotebookAccessError:
            raise EnvironmentError(
                f"Secret '{secret_name}' съществува, но notebook-ът няма достъп до него. "
                f"Включи 'Notebook access' от Colab Secrets панела."
            )

        if not value:
            raise EnvironmentError(f"Secret '{secret_name}' е празен.")

        keys[internal_name] = value

    return keys


API_KEYS = load_api_keys(SECRET_NAMES)



#TOOLS

In [ ]:
def ingest_pdf(pdf_path: str, chunk_size: int = 800, chunk_overlap: int = 100) -> int:
    """
    Чете PDF, разделя го на чънкове и ги записва в Chroma колекцията.
    Връща броя записани чънкове.
    """
    reader = PdfReader(pdf_path)
    full_text = "\n".join(page.extract_text() or "" for page in reader.pages)

    if not full_text.strip():
        raise ValueError(f"Не успях да извлека текст от '{pdf_path}' — възможно е да е сканиран PDF без текстов слой.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_text(full_text)

    ids = [f"chunk_{i}" for i in range(len(chunks))]
    collection.add(documents=chunks, ids=ids)

    print(f"Записани {len(chunks)} чънка от '{pdf_path}' в колекцията.")
    return len(chunks)


def retrieve_information(prompt: str) -> str:
    """
    Търси в Chroma колекцията най-релевантните чънкове спрямо prompt-а
    и връща отговор, генериран от LLM въз основа на тях.
    """
    results = collection.query(query_texts=[prompt], n_results=4)
    retrieved_chunks = results["documents"][0] if results["documents"] else []

    if not retrieved_chunks:
        return "Не намерих релевантна информация в документа за този въпрос."

    context = "\n\n---\n\n".join(retrieved_chunks)

    completion = client.chat.completions.create(
        model=MODELS["Gpt_4o_model"],
        messages=[
            {
                "role": "system",
                "content": (
                    "Отговаряй на въпроса САМО въз основа на предоставения контекст от документа. "
                    "Ако отговорът не се съдържа в контекста, кажи че информацията липсва."
                ),
            },
            {"role": "user", "content": f"Контекст:\n{context}\n\nВъпрос: {prompt}"},
        ],
    )

    return completion.choices[0].message.content

#TOOL DEFINITIONS

In [ ]:
retrieve_information_tool = {
    "type": "function",
    "function": {
        "name": "retrieve_information",
        "description": (
            "Търси информация в качения PDF документ (чрез векторна база с embeddings) "
            "и връща текстов отговор на конкретен въпрос към документа. "
            "Ползвай тази функция винаги когато потребителят иска информация, свързана със съдържанието на документа."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "prompt": {
                    "type": "string",
                    "description": "Въпросът, който да бъде зададен към документа.",
                }
            },
            "required": ["prompt"],
            "additionalProperties": False,
        },
    },
}


TOOLS = [
    retrieve_information_tool,
]


AVAILABLE_FUNCTIONS = {
    "retrieve_information": retrieve_information,
}

#STRUCTURED OUTPUT SCHEMAS


In [ ]:
class ResponseFormat(str, Enum):
    text = "text"
    image = "image"
    audio = "audio"


class AskAIRequest(BaseModel):
    """
    Схема за структурирания отговор на OpenAI — извлича от свободния
    въпрос на потребителя какъв е реалният prompt към документа
    и в какъв формат потребителят иска да получи отговора.
    """
    prompt: str = Field(
        description=(
            "Самият въпрос към документа, изчистен от всякакви указания за формат "
            "(напр. 'като снимка', 'на глас' и т.н.) — само същинския въпрос."
        )
    )
    format: ResponseFormat = Field(
        description=(
            "Предпочитаният от потребителя формат на отговора: "
            "'text' ако иска текстов отговор или не е уточнил, "
            "'image' ако иска отговорът визуализиран като изображение, "
            "'audio' ако иска отговорът прочетен на глас."
        )
    )

#Load Clients

In [ ]:
client = OpenAI(api_key=API_KEYS["OPENAI_API_KEY"])
elevenlabs_client = ElevenLabs(api_key=API_KEYS["ELEVENLABS_API_KEY"])

#Chromma client and collection
chroma_client = chromadb.Client()

openai_ef = OpenAIEmbeddingFunction(
    api_key=API_KEYS["OPENAI_API_KEY"],
    model_name=MODELS["Embedding_Model"],
)
collection = chroma_client.get_or_create_collection(
    name="document_chunks",
    embedding_function=openai_ef,
)